# Exploratory Data Analysis — PowerCo Customer Churn

---

**BCG X Data Science Job Simulation | Forage | July 2024**

**Author:** Ayush Seth | B.Sc. (Hons.) Statistics, Hindu College, University of Delhi

---

### Contents
1. Imports & Configuration
2. Load Data
3. Descriptive Statistics
4. Missing Values & Duplicates
5. Target Variable — Churn Distribution
6. Sales Channel Analysis
7. Consumption Analysis
8. Forecast Variable Analysis
9. Contract Type (Gas vs Electricity-Only)
10. Margin Analysis
11. Subscribed Power
12. Other Features (Products, Tenure, Origin)
13. Price Data Exploration
14. Skewness Summary
15. Save Cleaned Data

---

### Dataset
| File | Description | Size |
|------|-------------|------|
| `client_data.csv` | Client-level features: consumption, margins, contract dates | 14,606 rows × 26 cols |
| `price_data.csv` | Monthly energy prices per client (off-peak, peak, mid-peak) | 193,002 rows × 8 cols |

**Target:** `churn` — did the client churn in the next 3 months?  
**Class imbalance:** ~9.7% churn rate (1,419 churners out of 14,606 clients)

## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

%matplotlib inline

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi"     : 120,
    "axes.titlesize" : 13,
    "axes.labelsize" : 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

print("Libraries loaded ✓")

---
## 2. Load Data

We load `client_data.csv` and `price_data.csv` and immediately parse all date columns from strings to `datetime` objects — these will be used later for tenure and time-based feature engineering.

In [ ]:
client_df = pd.read_csv("client_data.csv")
price_df  = pd.read_csv("price_data.csv")

# Parse date columns upfront
DATE_COLS = ["date_activ", "date_end", "date_modif_prod", "date_renewal"]
for col in DATE_COLS:
    client_df[col] = pd.to_datetime(client_df[col], errors="coerce")

price_df["price_date"] = pd.to_datetime(price_df["price_date"], errors="coerce")

print(f"client_df : {client_df.shape[0]:,} rows × {client_df.shape[1]} columns")
print(f"price_df  : {price_df.shape[0]:,} rows × {price_df.shape[1]} columns")
print(f"Price date range: {price_df['price_date'].min().date()} → {price_df['price_date'].max().date()}")

### First look at client data

In [ ]:
client_df.head(3)

With the client data we have a mix of numeric and categorical data — transformations will be needed before modelling.

### First look at price data

In [ ]:
price_df.head(3)

The price data is purely numeric but contains many zeros — some clients may be on flat-rate tariffs without peak/mid-peak pricing.

---
## 3. Descriptive Statistics

### Data Types

In [ ]:
client_df.info()

Note: date columns have already been parsed to `datetime`. Categorical columns (`channel_sales`, `origin_up`, `has_gas`) will need encoding before modelling.

In [ ]:
price_df.info()

### Numeric Summary — Client Data

In [ ]:
client_df.describe().T

The key takeaway: percentile values reveal **highly skewed distributions** across consumption and forecast columns. The gap between 75th percentile and max values is enormous for variables like `cons_12m` and `net_margin`.

### Numeric Summary — Price Data

In [ ]:
price_df.describe().T

---
## 4. Missing Values & Duplicates

In [ ]:
client_missing = client_df.isnull().sum()
price_missing  = price_df.isnull().sum()

print("=== client_df — Missing Values ===")
missing_client = client_missing[client_missing > 0]
print(missing_client if len(missing_client) > 0 else "  None ✓")

print("\n=== price_df — Missing Values ===")
missing_price = price_missing[price_missing > 0]
print(missing_price if len(missing_price) > 0 else "  None ✓")

print(f"\nDuplicate rows — client_df : {client_df.duplicated().sum()}")
print(f"Duplicate rows — price_df  : {price_df.duplicated().sum()}")

Both datasets are clean with no missing values or duplicates.  
Note: `channel_sales` and `origin_up` contain a `MISSING` string that was added by the data team during pre-processing to flag original NaNs — this is a useful signal, not a data quality issue.

---
## 5. Target Variable — Churn Distribution

In [ ]:
def plot_stacked_bars(dataframe, title_, size_=(18, 10), rot_=0, legend_="upper right"):
    """Stacked percentage bar chart with value annotations."""
    ax = dataframe.plot(kind="bar", stacked=True, figsize=size_, rot=rot_, title=title_)
    for p in ax.patches:
        value = str(round(p.get_height(), 1))
        if value == "0.0":
            continue
        ax.annotate(
            value,
            ((p.get_x() + p.get_width() / 2) * 0.99 - 0.05,
             (p.get_y() + p.get_height() / 2) * 0.99),
            color="white", size=13,
        )
    plt.legend(["Retention", "Churn"], loc=legend_)
    plt.ylabel("Company base (%)")
    plt.tight_layout()
    plt.show()

n_retained   = (client_df["churn"] == 0).sum()
n_churned    = (client_df["churn"] == 1).sum()
n_total      = len(client_df)
pct_churned  = n_churned  / n_total * 100
pct_retained = n_retained / n_total * 100

print(f"Retained : {n_retained:,}  ({pct_retained:.1f}%)")
print(f"Churned  : {n_churned:,}  ({pct_churned:.1f}%)")
print("⚠  Class imbalance: ~9.7% positive class")
print("   → Handle with class_weight='balanced' or probability threshold tuning at modelling stage")

churn_percentage = pd.DataFrame({"Retention": [pct_retained], "Churn": [pct_churned]})
plot_stacked_bars(churn_percentage, "Churning Status", size_=(5, 5), legend_="lower right")

---
## 6. Sales Channel Analysis

In [ ]:
channel = (
    client_df.groupby(["channel_sales", "churn"])["id"]
    .count()
    .unstack(level=1)
    .fillna(0)
)
channel_churn_pct = (
    channel.div(channel.sum(axis=1), axis=0) * 100
).sort_values(by=1, ascending=False)

channel_churn_pct.rename(columns={0: "Retention%", 1: "Churn%"})

In [ ]:
plot_stacked_bars(channel_churn_pct, "Churn Rate by Sales Channel", rot_=30)

**Observations:**
- Churn is distributed across 5 channels; no single channel dominates churn rate
- The `MISSING` channel (imputed NaN values) has a **7.6% churn rate** — missing sales channel is a useful predictive signal in its own right
- The largest channel by volume drives the highest absolute number of churners despite a mid-range churn rate

---
## 7. Consumption Analysis

### Helper function for stacked histograms

In [ ]:
def plot_distribution(dataframe, column, ax, bins_=50):
    """Stacked histogram of retained vs churned for a numeric variable."""
    temp = pd.DataFrame({
        "Retention": dataframe[dataframe["churn"] == 0][column],
        "Churn"    : dataframe[dataframe["churn"] == 1][column],
    })
    temp.plot(kind="hist", bins=bins_, ax=ax, stacked=True, alpha=0.8)
    ax.set_xlabel(column)
    ax.ticklabel_format(style="plain", axis="x")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

### Consumption distributions

In [ ]:
consumption = client_df[[
    "id", "cons_12m", "cons_gas_12m", "cons_last_month", "imp_cons", "has_gas", "churn"
]]

fig, axs = plt.subplots(nrows=4, figsize=(18, 25))
fig.suptitle("Consumption — Distribution by Churn Status", fontsize=15, y=1.01)
plot_distribution(consumption,                               "cons_12m",        axs[0])
plot_distribution(consumption[consumption["has_gas"] == "t"], "cons_gas_12m",   axs[1])
plot_distribution(consumption,                               "cons_last_month",  axs[2])
plot_distribution(consumption,                               "imp_cons",         axs[3])
plt.tight_layout()
plt.show()

All consumption variables are **highly positively skewed** with a very long right-tail. The bulk of clients sit in a narrow low-consumption band, while a small number of large consumers pull the distribution out.

This skewness violates parametric model assumptions — log transformation will be applied in the Feature Engineering stage.

### Skewness values

In [ ]:
skew_consumption = {
    col: round(client_df[col].skew(), 2)
    for col in ["cons_12m", "cons_gas_12m", "cons_last_month", "imp_cons"]
}
pd.Series(skew_consumption, name="skewness").to_frame()

### Outlier detection — Boxplots

In [ ]:
fig, axs = plt.subplots(nrows=4, figsize=(18, 25))
fig.suptitle("Consumption — Outlier Detection (Boxplots)", fontsize=15, y=1.01)
sns.boxplot(x=consumption["cons_12m"],                                ax=axs[0])
sns.boxplot(x=consumption[consumption["has_gas"] == "t"]["cons_gas_12m"], ax=axs[1])
sns.boxplot(x=consumption["cons_last_month"],                          ax=axs[2])
sns.boxplot(x=consumption["imp_cons"],                                 ax=axs[3])

axs[0].set_xlim(-200_000,   2_000_000)
axs[1].set_xlim(-200_000,   2_000_000)
axs[2].set_xlim(-20_000,      100_000)
axs[3].set_xlim(
    consumption["imp_cons"].quantile(0.01),
    consumption["imp_cons"].quantile(0.99)
)
for ax in axs:
    ax.ticklabel_format(style="plain", axis="x")
plt.tight_layout()
plt.show()

Extreme outliers are visible in all consumption variables. These will be handled via log transformation and outlier capping during feature engineering.

---
## 8. Forecast Variable Analysis

In [ ]:
forecast_cols = [
    "forecast_cons_12m", "forecast_cons_year", "forecast_discount_energy",
    "forecast_meter_rent_12m", "forecast_price_energy_off_peak",
    "forecast_price_energy_peak", "forecast_price_pow_off_peak",
]

fig, axs = plt.subplots(nrows=len(forecast_cols), figsize=(18, 50))
fig.suptitle("Forecast Variables — Distribution by Churn Status", fontsize=15, y=1.005)
for ax, col in zip(axs, forecast_cols):
    plot_distribution(client_df, col, ax)
plt.tight_layout()
plt.show()

In [ ]:
pd.Series(
    {col: round(client_df[col].skew(), 2) for col in forecast_cols},
    name="skewness"
).to_frame()

Forecast variables mirror the consumption skewness pattern — same log transformation strategy will be applied across all of them.

---
## 9. Contract Type — Gas vs Electricity-Only

In [ ]:
gas_churn    = client_df[client_df["has_gas"] == "t"]["churn"].mean() * 100
no_gas_churn = client_df[client_df["has_gas"] == "f"]["churn"].mean() * 100
print(f"Gas clients (multi-product)   : {gas_churn:.1f}% churn")
print(f"Non-gas clients (electricity) : {no_gas_churn:.1f}% churn")
print(f"Difference                    : {no_gas_churn - gas_churn:.1f} percentage points")

In [ ]:
contract_type = client_df[["id", "has_gas", "churn"]]
contract = (
    contract_type
    .groupby(["churn", "has_gas"])["id"]
    .count()
    .unstack(level=0)
)
contract_pct = (contract.div(contract.sum(axis=1), axis=0) * 100).sort_values(by=1, ascending=False)
plot_stacked_bars(contract_pct, "Churn Rate by Contract Type (Gas vs Electricity-Only)")

Clients subscribed to **both gas and electricity** (multi-product) churn ~2% less. Higher switching costs from bundled services create stickiness — this `has_gas` flag will be a useful binary feature.

---
## 10. Margin Analysis

In [ ]:
margin = client_df[["id", "margin_gross_pow_ele", "margin_net_pow_ele", "net_margin"]]

pd.Series(
    {col: round(client_df[col].skew(), 2)
     for col in ["margin_gross_pow_ele", "margin_net_pow_ele", "net_margin"]},
    name="skewness"
).to_frame()

In [ ]:
fig, axs = plt.subplots(nrows=3, figsize=(18, 20))
fig.suptitle("Margin Variables — Outlier Detection (Boxplots)", fontsize=15)
sns.boxplot(x=margin["margin_gross_pow_ele"], ax=axs[0])
sns.boxplot(x=margin["margin_net_pow_ele"],   ax=axs[1])
sns.boxplot(x=margin["net_margin"],           ax=axs[2])
for ax in axs:
    ax.ticklabel_format(style="plain", axis="x")
plt.tight_layout()
plt.show()

`net_margin` has a **skewness of 36.6** with extreme outliers on the right tail — a handful of very high-margin clients dominate the distribution. This will need careful treatment (log transform + clipping) before modelling.

Later, the Random Forest model will confirm that **net_margin is the single most important predictor of churn** — making this variable critical to get right.

---
## 11. Subscribed Power

In [ ]:
power = client_df[["id", "pow_max", "churn"]]
fig, ax = plt.subplots(figsize=(18, 7))
plot_distribution(power, "pow_max", ax)
ax.set_title("Subscribed Power — Distribution by Churn Status")
plt.tight_layout()
plt.show()

print(f"pow_max skewness: {client_df['pow_max'].skew():.2f}  → log transform candidate")

---
## 12. Other Features

### Number of Active Products

In [ ]:
others = client_df[["id", "nb_prod_act", "num_years_antig", "origin_up", "churn"]]

products = (
    others.groupby(["nb_prod_act", "churn"])["id"]
    .count().unstack(level=1)
)
products_pct = (products.div(products.sum(axis=1), axis=0) * 100).sort_values(by=1, ascending=False)
plot_stacked_bars(products_pct, "Churn Rate by Number of Active Products")

### Client Tenure (Years of Antiquity)

In [ ]:
years_antig = (
    others.groupby(["num_years_antig", "churn"])["id"]
    .count().unstack(level=1)
)
years_antig_pct = years_antig.div(years_antig.sum(axis=1), axis=0) * 100
plot_stacked_bars(years_antig_pct, "Churn Rate by Client Tenure (Years)")

In [ ]:
client_df.groupby("num_years_antig")["churn"] \
    .mean() \
    .mul(100) \
    .round(1) \
    .rename("churn_%") \
    .to_frame()

Clients active for fewer years tend to churn more — newer clients are less loyal. This pattern motivates the `tenure` and `months_activ` features in the Feature Engineering stage.

### Origin Campaign

In [ ]:
origin = (
    others.groupby(["origin_up", "churn"])["id"]
    .count().unstack(level=1)
)
origin_pct = origin.div(origin.sum(axis=1), axis=0) * 100
plot_stacked_bars(origin_pct, "Churn Rate by Origin Campaign", rot_=30)

---
## 13. Price Data Exploration

In [ ]:
print(f"Unique clients in price data : {price_df['id'].nunique():,}")
print(f"Unique months covered        : {price_df['price_date'].nunique()}")
print(f"Date range                   : {price_df['price_date'].min().date()} → {price_df['price_date'].max().date()}")

### Distribution of Variable and Fixed Prices

In [ ]:
price_var_cols = ["price_off_peak_var", "price_peak_var", "price_mid_peak_var"]
price_fix_cols = ["price_off_peak_fix", "price_peak_fix", "price_mid_peak_fix"]

fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
fig.suptitle("Price Data — Distribution of Variable and Fixed Prices", fontsize=14)
for ax, col in zip(axs[0], price_var_cols):
    sns.histplot(price_df[col], bins=50, ax=ax, kde=True)
    ax.set_title(col)
    ax.ticklabel_format(style="plain", axis="x")
for ax, col in zip(axs[1], price_fix_cols):
    sns.histplot(price_df[col], bins=50, ax=ax, kde=True)
    ax.set_title(col)
    ax.ticklabel_format(style="plain", axis="x")
plt.tight_layout()
plt.show()

### Zero-value Analysis

In [ ]:
zero_counts = pd.DataFrame([{
    "column"  : col,
    "zeros"   : (price_df[col] == 0).sum(),
    "zero_pct": round((price_df[col] == 0).mean() * 100, 1)
} for col in price_var_cols + price_fix_cols])

zero_counts.set_index("column")

Many zeros in peak and mid-peak columns — a significant portion of clients are on **flat off-peak-only tariffs**. This explains the BCG hypothesis that price sensitivity (specifically off-peak price changes) could be a churn driver.

### Monthly Average Price Trends — 2015

In [ ]:
monthly_avg = price_df.groupby("price_date")[price_var_cols + price_fix_cols].mean()

fig, axs = plt.subplots(nrows=2, figsize=(14, 10))
fig.suptitle("Monthly Average Prices — 2015", fontsize=14)

for col in price_var_cols:
    axs[0].plot(monthly_avg.index, monthly_avg[col], marker="o", label=col)
axs[0].set_title("Variable Prices (energy component)")
axs[0].legend()
axs[0].set_ylabel("Price")

for col in price_fix_cols:
    axs[1].plot(monthly_avg.index, monthly_avg[col], marker="o", label=col)
axs[1].set_title("Fixed Prices (power component)")
axs[1].legend()
axs[1].set_ylabel("Price")
plt.tight_layout()
plt.show()

The Dec–Jan price difference feature (engineered in the next notebook) is motivated by this chart — off-peak variable prices shift noticeably across the year, and that year-on-year delta is expected to correlate with churn behaviour.

---
## 14. Skewness Summary

A consolidated view of all numeric features that require transformation before modelling.

In [ ]:
skew_cols = [
    "cons_12m", "cons_gas_12m", "cons_last_month", "imp_cons",
    "forecast_cons_12m", "forecast_cons_year", "forecast_discount_energy",
    "forecast_meter_rent_12m", "forecast_price_energy_off_peak",
    "forecast_price_energy_peak", "forecast_price_pow_off_peak",
    "net_margin", "margin_gross_pow_ele", "margin_net_pow_ele", "pow_max",
]

skewness = (
    client_df[skew_cols]
    .skew()
    .sort_values(ascending=False)
    .rename("skewness")
    .to_frame()
)
skewness["action"] = skewness["skewness"].apply(
    lambda s: "✗ log transform" if abs(s) > 1 else "✓ ok"
)

print(f"Features requiring log transform: {(skewness['action'] == '✗ log transform').sum()} / {len(skewness)}")
skewness

---
## 15. Save Cleaned Data

Apply a consistent `MISSING` label for any remaining NaN values in categorical columns, then save for use in Feature Engineering.

In [ ]:
# Fill NaN in categorical columns with 'MISSING'
for col in ["channel_sales", "origin_up"]:
    client_df[col] = client_df[col].fillna("MISSING")

# Verify no nulls remain
assert client_df.isnull().sum().sum() == 0, "Unexpected nulls remain"

client_df.to_csv("clean_data_after_eda.csv", index=False)
print(f"✓ clean_data_after_eda.csv saved — {client_df.shape[0]:,} rows × {client_df.shape[1]} columns")
print("\nEDA complete → proceed to Feature Engineering notebook")

---
## Summary of Key EDA Findings

| Finding | Implication for Modelling |
|---------|--------------------------|
| ~9.7% churn rate | Class imbalance — use `class_weight='balanced'` or tune decision threshold |
| All consumption & forecast cols heavily right-skewed | Log-transform before modelling |
| `net_margin` skewness = 36.6 | Critical feature — handle carefully; will be top predictor |
| `MISSING` sales channel has 7.6% churn | Missing channel is predictive, not just noise |
| Gas clients churn 2% less | Multi-product loyalty signal — binary flag useful |
| Short-tenure clients churn more | Derive `tenure` and `months_activ` features |
| Many zeros in peak/mid-peak prices | Flat-rate tariff segment — Dec-Jan price diff feature motivated |

→ Next: `Feature_Engineering.ipynb`